In [9]:
import pandas as pd
import numpy as np
import torch
import requests
from PIL import Image
from io import BytesIO
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [10]:
import torch
import torchvision.models as models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet50(pretrained=True)
model = torch.nn.Sequential(*list(model.children())[:-1])

model = model.to(device)
model.eval()

torch.backends.cudnn.benchmark = True

In [11]:
import torchvision.transforms as transforms
preprocess = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [12]:
session = requests.Session()

def load_image(url):

    try:
        r = session.get(url, timeout=7)

        if r.status_code != 200:
            return None

        img = Image.open(BytesIO(r.content)).convert("RGB")
        img = preprocess(img)

        return img

    except:
        return None

In [13]:
def load_images_parallel(urls, ids, workers=32):

    images = []
    valid_ids = []

    with ThreadPoolExecutor(max_workers=workers) as executor:
        results = list(executor.map(load_image, urls))

    for img, pid in zip(results, ids):
        if img is not None:
            images.append(img)
            valid_ids.append(pid)

    return images, valid_ids

In [14]:
from tqdm import tqdm

def generate_embeddings(urls, ids, chunk_id, batch_size=256):

    embeddings = []
    valid_ids = []

    total_batches = (len(urls) + batch_size - 1) // batch_size

    for start in tqdm(
        range(0, len(urls), batch_size),
        total=total_batches,
        desc=f"Chunk {chunk_id}",
        unit="batch"
    ):

        batch_urls = urls[start:start+batch_size]
        batch_ids = ids[start:start+batch_size]

        images, good_ids = load_images_parallel(batch_urls, batch_ids)

        if len(images) == 0:
            continue

        batch = torch.stack(images).to(device)

        with torch.no_grad():
            emb = model(batch)

        emb = emb.view(emb.size(0), -1).cpu().numpy()

        embeddings.append(emb)
        valid_ids.extend(good_ids)

    if len(embeddings) == 0:
        return None, None

    embeddings = np.concatenate(embeddings, axis=0)

    return embeddings, np.array(valid_ids)

In [37]:
import pandas as pd
import numpy as np
import os

reader = pd.read_csv("usa_europe_geotagged.csv", chunksize=10000)

EMB_DIR = "embeddings"
os.makedirs(EMB_DIR, exist_ok=True)

START_CHUNK = 693

for i, chunk in enumerate(reader):

    if i < START_CHUNK:
        continue

    urls = chunk["downloadurl"].tolist()
    ids = chunk["photoid"].tolist()

    embeddings, valid_ids = generate_embeddings(urls, ids, i)

    if embeddings is None:
        print(f"Chunk {i}: no valid images")
        continue

    np.save(f"{EMB_DIR}/embeddings_chunk_{i}.npy", embeddings)
    np.save(f"{EMB_DIR}/ids_chunk_{i}.npy", valid_ids)

Chunk 767: 100%|██████████| 10/10 [00:45<00:00,  4.53s/batch]


In [15]:
import pandas as pd
import numpy as np
import os

reader = pd.read_csv("usa_europe_geotagged.csv", chunksize=10000)

EMB_DIR = "embeddings"
os.makedirs(EMB_DIR, exist_ok=True)

START_CHUNK = 390
END_CHUNK = 392

for i, chunk in enumerate(reader):

    if i < START_CHUNK:
        continue
    if i > END_CHUNK:
        break

    urls = chunk["downloadurl"].tolist()
    ids = chunk["photoid"].tolist()

    embeddings, valid_ids = generate_embeddings(urls, ids, i)

    if embeddings is None or len(valid_ids) == 0:
        print(f"Chunk {i}: no valid images")
        continue

    np.save(f"{EMB_DIR}/embeddings_chunk_{i}.npy", embeddings)
    np.save(f"{EMB_DIR}/ids_chunk_{i}.npy", valid_ids)

    print(f"Chunk {i} saved ({len(valid_ids)} embeddings)")

Chunk 390: 100%|██████████| 40/40 [01:30<00:00,  2.27s/batch]


Chunk 390 saved (8516 embeddings)


Chunk 391: 100%|██████████| 40/40 [02:49<00:00,  4.24s/batch]


Chunk 391 saved (7326 embeddings)


Chunk 392: 100%|██████████| 40/40 [01:28<00:00,  2.21s/batch]


Chunk 392 saved (8934 embeddings)
